___
# <center>Interpretabilidade e justiça algorítmica</center>
___

## Aula 25

Nas aulas 23 e 24, o objetivo foi **prever bem**: a floresta ganhou da
regressão logística na AUC. O preço é que a floresta não tem coeficientes para
ler. Hoje respondemos a duas perguntas que aparecem assim que um modelo desses
é usado de verdade:

1. **Por que o modelo deu essa probabilidade para este processo?** A resposta é
   o **SHAP**, que reparte a previsão entre as variáveis.
2. **O modelo erra do mesmo jeito para todo mundo?** A resposta é medir os erros
   **por grupo**, como numa auditoria.

As duas perguntas têm consequência jurídica: a LGPD (art. 20) dá a quem é
afetado por uma decisão automatizada o direito de pedir revisão e de saber os
critérios usados.

A aula é curta. Faça as partes 1 a 3 em aula; o que faltar fica para casa.

___
<div id="indice"></div>

## Índice

- [Preparação: pacotes e base](#base)
- [1. A floresta e os valores SHAP](#parte1)
- [2. A cascata: a explicação de um processo](#parte2)
- [3. O enxame: as explicações de muitos processos](#parte3)
- [4. Erros por grupo na triagem do NatJus](#parte4)
- [5. A explicação basta para a LGPD?](#parte5)
- [Para o Projeto 08](#p08)

___
<div id="base"></div>

# Preparação: pacotes e base

O pacote novo é o **`shap`**. No Colab ele já vem instalado; se a importação der
erro, rode antes `%pip install shap`.

A base é a das aulas 21 e 22: 1.000 ações revisionais simuladas, com a
**sentença favorável ao autor** como resposta. Você já conhece as colunas: taxa
de juros, valor da causa, valor financiado, idade do autor, perícia, juizado,
escritório de massa e tribunal. A coluna `valor_condenacao` fica de fora: ela só
existe quando a sentença é favorável, então seria **vazamento** (aula 24).

In [ ]:
# pandas: tabelas; plotnine: gráficos
import pandas as pd
from plotnine import *

# a floresta e a validação cruzada, como na aula 24
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# shap: os valores de Shapley para modelos de machine learning
import shap

In [ ]:
base = pd.read_csv("https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados/processos_acordo.csv")

# a resposta e as variáveis; o tribunal vira colunas 0/1 (uma por tribunal)
y = base["sentenca_favoravel"]
X = base.drop(columns=["processo", "sentenca_favoravel", "valor_condenacao"])
X = pd.get_dummies(X, columns=["tribunal"], dtype=float)
print(X.shape)
X.head(3)

[Volta ao Índice](#indice)

___
<div id="parte1"></div>

# 1. A floresta e os valores SHAP

Primeiro, o modelo: uma floresta de 300 árvores, com pelo menos 10 processos em
cada folha. A validação cruzada mostra que ele prevê bem.

In [ ]:
floresta = RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=1)
pastas = StratifiedKFold(5, shuffle=True, random_state=25)
print("AUC na validação cruzada:", cross_val_score(floresta, X, y, cv=pastas, scoring="roc_auc").mean().round(3))

floresta.fit(X, y)   # o modelo final, ajustado em toda a base

**Como o SHAP calcula.** Nos slides, com duas variáveis, a contribuição de cada
uma foi a média do quanto ela mexe na previsão quando é "revelada", em todas as
ordens possíveis. "Revelar" uma variável quer dizer fixar o valor do processo
nela e deixar as outras variarem como numa **amostra de fundo** (processos de
referência). Com muitas variáveis, o pacote `shap` faz essa mesma conta de forma
rápida para árvores e florestas.

Três escolhas na célula abaixo:

- `fundo`: 100 processos sorteados, a referência do "sem saber nada";
- `model_output`: a escala da explicação. Queremos as contribuições em
  **probabilidade**, e não em log-chance;
- `[:, :, 1]`: a floresta dá duas probabilidades (desfavorável e favorável);
  ficamos com a da sentença **favorável**.

Complete a escala e rode. A célula leva alguns segundos.

In [ ]:
fundo = X.sample(100, random_state=3)
explicador = shap.TreeExplainer(floresta, fundo, model_output="____")

amostra = X.sample(300, random_state=4)        # 300 processos para explicar
valores = explicador(amostra)[:, :, 1]          # contribuições para a sentença favorável
valores.shape

O resultado tem uma linha por processo e uma coluna por variável: é o quanto
cada variável empurrou a previsão daquele processo para cima ou para baixo.

**A propriedade que faz o SHAP útil:** para cada processo, a média do modelo
(`base_values`) mais a soma das contribuições dá **exatamente** a previsão. A
célula abaixo confere isso no primeiro processo da amostra.

In [ ]:
primeiro = valores[0]
print("média do modelo (base):  ", round(primeiro.base_values, 3))
print("soma das contribuições:  ", round(primeiro.values.sum(), 3))
print("base + soma:             ", round(primeiro.base_values + primeiro.values.sum(), 3))
print("previsão da floresta:    ", round(floresta.predict_proba(amostra.iloc[[0]])[0, 1], 3))

[Volta ao Índice](#indice)

___
<div id="parte2"></div>

# 2. A cascata: a explicação de um processo

A cascata (`waterfall`) começa na média do modelo, embaixo, e soma uma variável
por vez até chegar à previsão, em cima. Barras vermelhas empurram para a
sentença favorável; azuis, contra. Ao lado de cada variável aparece o valor dela
naquele processo.

Escolha um processo da amostra pela posição (de 0 a 299) e rode. Comece pelo
processo 0 e depois experimente outros.

In [ ]:
shap.plots.waterfall(valores[____])

**Pergunta 1.** No processo que você escolheu, qual variável mais empurrou a
previsão, e para que lado? Escreva a explicação em uma frase que um advogado
entenderia, sem usar a palavra SHAP.

[Volta ao Índice](#indice)

___
<div id="parte3"></div>

# 3. O enxame: as explicações de muitos processos

O enxame (`beeswarm`) põe todos os 300 processos num gráfico só. Cada linha é uma
variável; cada ponto, um processo. A posição é a contribuição (o SHAP) e a cor é
o valor da variável: vermelho, alto; azul, baixo.

In [ ]:
shap.plots.beeswarm(valores)

A média do tamanho das contribuições, sem o sinal, resume a importância de cada
variável. É a mesma pergunta da importância da aula 24, respondida de outro
jeito. A célula abaixo põe as duas lado a lado.

In [ ]:
importancia = pd.DataFrame({
    "SHAP (média sem sinal)": abs(pd.DataFrame(valores.values, columns=X.columns)).mean(),
    "importância da floresta": floresta.feature_importances_,
}, index=X.columns)
importancia.sort_values("SHAP (média sem sinal)", ascending=False).round(3)

**Pergunta 2.** No enxame, para que lado o **escritório de massa** empurra a
previsão? Isso quer dizer que, se o autor trocasse de escritório, a chance de
ganhar mudaria? Justifique.

[Volta ao Índice](#indice)

___
<div id="parte4"></div>

# 4. Erros por grupo na triagem do NatJus

Agora a outra pergunta: o modelo erra do mesmo jeito para todo mundo? Voltamos à
triagem do **Projeto 06**: 1.000 processos de saúde, a probabilidade que o modelo
deu a cada um (`prob`) e se o caso precisava mesmo de análise aprofundada
(`precisa_analise`). O grupo é **quem é o réu**: o poder público (SUS) ou uma
operadora de plano de saúde (`reu_plano_saude`).

Complete a função: a sensibilidade é a proporção, entre os casos que
**precisavam** de análise, dos que foram **marcados**.

In [ ]:
triagem = pd.read_csv("https://jtrecenti.github.io/cdad2-202662/ex06-onde-colocar-o-corte/dados/triagem_natjus.csv")
triagem["reu"] = triagem["reu_plano_saude"].map({0: "poder público", 1: "plano de saúde"})


def por_grupo(dados, corte):
    """Para cada grupo de réu: quantos processos, quantos marcados, sensibilidade e precisão."""
    linhas = []
    for reu, d in dados.groupby("reu"):
        marcado = d["prob"] >= corte
        precisa = d["precisa_analise"] == 1
        linhas.append({
            "réu": reu,
            "processos": len(d),
            "marcados (%)": marcado.mean(),
            "sensibilidade": (marcado & precisa).sum() / ____.sum(),
            "precisão": (marcado & precisa).sum() / marcado.sum(),
        })
    return pd.DataFrame(linhas).round(2)


por_grupo(triagem, corte=0.20)   # a mediana dos cortes da turma no Projeto 06

Um corte **por grupo** permite igualar uma das medidas. A célula abaixo testa
cortes diferentes para quem processa um plano de saúde, mantendo 0,20 para quem
processa o poder público. Procure o corte que deixa a sensibilidade dos dois
grupos parecida, e olhe o que acontece com a precisão.

In [ ]:
publico = por_grupo(triagem, corte=0.20).query("réu == 'poder público'")
for corte_plano in [0.20, 0.19, 0.18, 0.17, 0.15]:
    plano = por_grupo(triagem, corte=corte_plano).query("réu == 'plano de saúde'")
    print(f"corte do plano {corte_plano:.2f}: sensibilidade {plano['sensibilidade'].item():.2f} "
          f"(poder público {publico['sensibilidade'].item():.2f}), "
          f"precisão {plano['precisão'].item():.2f} (poder público {publico['precisão'].item():.2f})")

**Pergunta 3.** Usar cortes diferentes por grupo é justo? Dê um argumento a favor
(pensando em quem precisa de análise e não é encontrado) e um contra (pensando
em tratar casos parecidos do mesmo jeito). Os grupos aqui são pequenos (61 e 28
casos que precisavam de análise): o que isso muda na conclusão?

[Volta ao Índice](#indice)

___
<div id="parte5"></div>

# 5. A explicação basta para a LGPD?

O art. 20 da LGPD diz que o titular dos dados pode pedir a **revisão** de
decisões tomadas unicamente com base em tratamento automatizado que afetem seus
interesses, e o § 1º obriga o controlador a fornecer, quando solicitadas,
**informações claras e adequadas** sobre os critérios e os procedimentos usados,
observados os segredos comercial e industrial.

**Pergunta 4.** Um autor cujo processo não foi marcado para análise aprofundada
pede, com base no art. 20, para saber por quê. O tribunal entrega a cascata do
SHAP do processo dele. Isso cumpre o § 1º? O que mais você pediria ao tribunal,
pensando na parte 4?

[Volta ao Índice](#indice)

___
<div id="p08"></div>

# Para o Projeto 08

O SHAP funciona com o seu modelo do projeto. Como o modelo é um `pipeline` (o
preparo e depois a floresta), a explicação é feita sobre as colunas **já
preparadas**. O esqueleto abaixo mostra o caminho; troque `modelo` e `X_treino`
pelos nomes do seu notebook.

```python
preparo, floresta = modelo[0], modelo[-1]
X_preparado = preparo.transform(X_treino)
explicador = shap.TreeExplainer(floresta, X_preparado.sample(100, random_state=1),
                                model_output="probability")
valores = explicador(X_preparado.sample(300, random_state=2))[:, :, 1]
shap.plots.beeswarm(valores)
```

Se uma variável dominar o enxame muito mais do que você esperava, pergunte de
novo, como na aula 24: **no dia da previsão, eu já sei isto?**

[Volta ao Índice](#indice)